In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import gpytorch
import networkx as nx
from tqdm.auto import tqdm
from grf_gp.sampler import GRFSampler
from grf_gp.utils.spectral import get_normalized_laplacian
from grf_gp.kernels.diffusion import DiffusionGRFKernel
from grf_gp.kernels.general import GeneralGRFKernel
from grf_gp.kernels.base import BaseExactKernel
from grf_gp.model import GRFGP, ExactGraphGP
project_root = os.path.abspath("../../..")
sys.path.append(project_root)
from traffic_utils.preprocessing import load_PEMS
from experiments.regression.utils import train_model, evaluate_model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)


In [ ]:
SEEDS = [0, 2]
WALKS_PER_NODE_LIST = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
NUM_TRAIN = 250
MAX_WALK_LENGTH = 3
P_HALT = 0.5
N_PROCESSES = 4
EXACT_LR = 0.05
EXACT_STEPS = 100
GRF_LR = 0.05
GRF_STEPS = 1000


In [ ]:
class ExactDiffusionKernel(BaseExactKernel):
    def __init__(self, L, beta_max=2.0, **kwargs):
        super().__init__(**kwargs)
        self.register_buffer("L", L)
        self.beta_max = beta_max # prevent oversmoothing

        self.register_parameter(
            name="raw_beta", parameter=torch.nn.Parameter(torch.tensor(1.0))
        )
        self.register_parameter(
            name="raw_sigma_f", parameter=torch.nn.Parameter(torch.tensor(1.0))
        )

    @property
    def beta(self):
        return self.beta_max * torch.sigmoid(self.raw_beta)   # in (0, beta_max)

    @property
    def sigma_f(self):
        return torch.nn.functional.softplus(self.raw_sigma_f)

    def _full_kernel_matrix(self) -> torch.Tensor:
        return self.sigma_f**2 * torch.matrix_exp(-self.beta * self.L)


In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_data_for_seed(seed):
    set_seed(seed)
    G, data_train, data_test, data = load_PEMS(num_train=NUM_TRAIN, path=os.path.join(project_root, "data/traffic"))
    x_train, y_train = data_train
    x_test, y_test = data_test
    _, _ = data
    orig_mean = np.mean(y_train)
    orig_std = np.std(y_train)
    y_train_norm = (y_train - orig_mean) / orig_std
    y_test_norm = (y_test - orig_mean) / orig_std
    X_train = torch.tensor(x_train).long().to(device)
    Y_train = torch.tensor(y_train_norm).to(device).flatten()
    X_test = torch.tensor(x_test).long().to(device)
    Y_test = torch.tensor(y_test_norm).to(device).flatten()
    adjacency_matrix = nx.to_numpy_array(G)
    L = get_normalized_laplacian(adjacency_matrix)
    L = torch.tensor(L).to(device)
    return X_train, Y_train, X_test, Y_test, L, float(orig_std)



def run_exact(seed, X_train, Y_train, X_test, Y_test, L, orig_std):
    set_seed(seed)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    kernel = ExactDiffusionKernel(L).to(device)
    model = ExactGraphGP(X_train, Y_train, likelihood, kernel).to(device)
    train_model(model, likelihood, X_train, Y_train, lr=EXACT_LR, max_iter=EXACT_STEPS, print_every=1)
    lml, rmse, nlpd = evaluate_model(model, likelihood, X_train, Y_train, X_test, Y_test, orig_std)
    return {"seed": seed, "model": "ExactDiffusion", "walks_per_node": np.nan, "lml": lml, "rmse": rmse, "nlpd": nlpd}

def run_grf(seed, model_name, walks_per_node, rw_mats, X_train, Y_train, X_test, Y_test, L, orig_std):
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    if model_name == "GRFPolynomial":
        kernel = GeneralGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
    else:
        kernel = DiffusionGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
    model = GRFGP(X_train, Y_train, likelihood, kernel).to(device)
    train_model(model, likelihood, X_train, Y_train, lr=GRF_LR, max_iter=GRF_STEPS, print_every=1)
    lml, rmse, nlpd = evaluate_model(model, likelihood, X_train, Y_train, X_test, Y_test, orig_std)
    return {"seed": seed, "model": model_name, "walks_per_node": int(walks_per_node), "lml": lml, "rmse": rmse, "nlpd": nlpd}


In [ ]:
results = []
for seed in tqdm(SEEDS, desc="Seeds"):
    X_train, Y_train, X_test, Y_test, L, orig_std = load_data_for_seed(seed)
    print("Running Exact Diffusion Kernel")
    results.append(run_exact(seed, X_train, Y_train, X_test, Y_test, L, orig_std))
    for walks_per_node in tqdm(WALKS_PER_NODE_LIST, leave=False, desc=f"Seed {seed} walkers"):
        sampler = GRFSampler(L, walks_per_node, P_HALT, MAX_WALK_LENGTH, seed=seed, use_tqdm=False, n_processes=N_PROCESSES)
        print(f"Sampling random walk matrices for {walks_per_node} walks/node")
        rw_mats = sampler.sample_random_walk_matrices()
        print(f"Running GRFPolynomial with {walks_per_node} walks/node")
        results.append(run_grf(seed, "GRFPolynomial", walks_per_node, rw_mats, X_train, Y_train, X_test, Y_test, L, orig_std))
        print(f"Running GRFDiffusion with {walks_per_node} walks/node")
        results.append(run_grf(seed, "GRFDiffusion", walks_per_node, rw_mats, X_train, Y_train, X_test, Y_test, L, orig_std))
results_df = pd.DataFrame(results)
results_dir = os.path.join(project_root, "experiments/regression/traffic_prediction/results")
os.makedirs(results_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(results_dir, f"traffic_regression_seed_walk_sweep_{timestamp}.csv")
results_df.to_csv(csv_path, index=False)
print(csv_path)
print(results_df.shape)
results_df.head()
